In [1]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("lmms-lab/AISG_Challenge")

/home/incomple_/anaconda3/envs/llamafactory/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(ds)

DatasetDict({
    test: Dataset({
        features: ['qid', 'video_id', 'question_type', 'capability', 'question', 'duration', 'question_prompt', 'answer', 'youtube_url'],
        num_rows: 1500
    })
})


In [3]:
import pandas as pd
# save ds as a csv
df = pd.DataFrame(ds['test'])
df

,qid,video_id,question_type,capability,question,duration,question_prompt,answer,youtube_url
0,0008-0,sj81PWrerDk,Primary Open-ended Question,Plot Attribute (Montage),What is the difference between the action of t...,8.85,Please state your answer with a brief explanat...,,https://www.youtube.com/shorts/sj81PWrerDk
1,0008-1,sj81PWrerDk,Paraphrased Open-ended Question,Plot Attribute (Montage),Can you describe how the actions of the last p...,8.85,Please state your answer with a brief explanat...,,https://www.youtube.com/shorts/sj81PWrerDk
2,0008-2,sj81PWrerDk,Correctly-led Open-ended Question,Plot Attribute (Montage),Did the last person open the bottle without us...,8.85,Please state your answer with a brief explanat...,,https://www.youtube.com/shorts/sj81PWrerDk
3,0008-3,sj81PWrerDk,Wrongly-led Open-ended Question,Plot Attribute (Montage),Did the last person in the video open the bott...,8.85,Please state your answer with a brief explanat...,,https://www.youtube.com/shorts/sj81PWrerDk
4,0008-7,sj81PWrerDk,Multiple-choice Question with a Single Correct...,Plot Attribute (Montage),How does the last person in the video open the...,8.85,E. None of the above\nSelect one best answer t...,,https://www.youtube.com/shorts/sj81PWrerDk
...,...,...,...,...,...,...,...,...,...
1495,1344-0,eLJNa61S4RE,Primary Open-ended Question,Character Reaction Causality,"In this video, why does the child smile after ...",18.35,Please state your answer with a brief explanat...,,https://www.youtube.com/shorts/eLJNa61S4RE
1496,1344-1,eLJNa61S4RE,Paraphrased Open-ended Question,Character Reaction Causality,What causes the child to smile after receiving...,18.35,Please state your answer with a brief explanat...,,https://www.youtube.com/shorts/eLJNa61S4RE
1497,1344-2,eLJNa61S4RE,Correctly-led Open-ended Question,Character Reaction Causality,Does the child smile after the injection becau...,18.35,Please state your answer with a brief explanat...,,https://www.youtube.com/shorts/eLJNa61S4RE
1498,1344-3,eLJNa61S4RE,Wrongly-led Open-ended Question,Character Reaction Causality,Is the child's smile after the injection due t...,18.35,Please state your answer with a brief explanat...,,https://www.youtube.com/shorts/eLJNa61S4RE


In [ ]:
# save df
df.to_csv('challenge_data.csv', index=False, encoding='utf-8-sig')

# Import youtube data

In [ ]:
import os
import pandas as pd
# from pytube import YouTube
from pytubefix import YouTube
from pytubefix.cli import on_progress

# Example: assume you already have df loaded with columns: ['youtube_url', 'duration']

# 1. Drop duplicate URLs
df_unique = df.drop_duplicates(subset=['youtube_url']).reset_index(drop=True)

# Lists to collect successful data and errors
video_data = []
failed_urls = []

video_folder = 'videos_full'
os.makedirs(video_folder, exist_ok=True)

# 2. Iterate over each unique URL
for idx, row in df_unique.iterrows():
    yt_url = row['youtube_url']
    duration = row['duration']  # from your original DataFrame
    video_id =  yt_url.strip().split("/")[-1]
    video_path = os.path.join(video_folder, f"{video_id}.mp4")

    try:
        yt = YouTube(yt_url, on_progress_callback=on_progress)

        # Collect metadata in order of importance
        title         = yt.title
        description   = yt.description
        # 2a. Retrieve captions in a fallback manner
        caption_text = ''
        caption_lang = ''
        if yt.captions:  # If there are any captions at all
            # Prefer English if available
            if 'en' in yt.captions:
                caption_text = yt.captions['en'].generate_srt_captions()
                caption_lang = 'en'
            elif 'a.en' in yt.captions:
                caption_text = yt.captions['a.en'].generate_srt_captions()
                caption_lang = 'a.en'
            else:
                # Fallback: pick the first available language
                first_lang = list(yt.captions.keys())[0].code
                caption_text = yt.captions[first_lang].generate_srt_captions()
                caption_lang = first_lang
                
        publish_date  = yt.publish_date
        rating        = yt.rating
        channel_id    = yt.channel_id
        channel_url   = yt.channel_url
        thumbnail_url = yt.thumbnail_url
        channel_name  = yt.author
        views         = yt.views
        keywords      = yt.keywords  # list of strings
        video_id      = yt.video_id

        # 3. Download the video
        
        # stream = yt.streams.get_highest_resolution()
        # video_path = stream.download(output_path=download_folder)
        # Change video_path to the relative path
        # path = download_folder + '/' + video_id


        # 4. Store metadata
        info = {
            'youtube_url':   yt_url,
            'title':         title,
            'description':   description,
            'caption':       caption_text,
            'caption_lang':  caption_lang,
            'publish_date':  publish_date,
            'rating':        rating,
            'channel_id':    channel_id,
            'channel_url':   channel_url,
            'thumbnail_url': thumbnail_url,
            'channel_name':  channel_name,
            'views':         views,
            'keywords':      keywords,
            'duration':      duration,
            'video_path':    video_path
        }

        video_data.append(info)
        print(f"Processed URL {yt_url} successfully.")

    except Exception as e:
        # If anything fails, capture the URL and reason
        error_info = {
            'youtube_url': yt_url,
            'error': str(e)
        }
        failed_urls.append(error_info)
        
        info = {
            'youtube_url':   yt_url,
            'title':         None,
            'description':   None,
            'caption':       None,
            'caption_lang':  None,
            'publish_date':  None,
            'rating':        None,
            'channel_id':    None,
            'channel_url':   None,
            'thumbnail_url': None,
            'channel_name':  None,
            'views':         None,
            'keywords':      None,
            'duration':      None,
            'video_path':    video_path
        }
        print(f"Failed to process URL {yt_url}: {e}")
        video_data.append(info)

# 5. Create DataFrames from successful and failed entries
metadata_df = pd.DataFrame(video_data)
failed_df = pd.DataFrame(failed_urls)

# 6. Save both DataFrames to CSV in the same location
metadata_df.to_csv('video_metadata.csv', index=False, encoding='utf-8-sig')
failed_df.to_csv('video_errors.csv', index=False, encoding='utf-8-sig')

print("Finished processing.")
print(f"Successfully processed: {len(metadata_df)} videos.")
print(f"Failed to process: {len(failed_df)} videos. Check 'video_errors.csv' for details.")


/tmp/ipykernel_28718/986113750.py:27: DeprecationWarning: Call to deprecated function get_by_language_code (This object can be treated as a dictionary, i.e. captions['en']).
  caption_obj   = yt.captions.get_by_language_code('en')


Failed to process URL https://www.youtube.com/shorts/AGCyLqLuUJ0: AGCyLqLuUJ0 is unavailable


/tmp/ipykernel_28718/986113750.py:27: DeprecationWarning: Call to deprecated function get_by_language_code (This object can be treated as a dictionary, i.e. captions['en']).
  caption_obj   = yt.captions.get_by_language_code('en')


/tmp/ipykernel_28718/986113750.py:27: DeprecationWarning: Call to deprecated function get_by_language_code (This object can be treated as a dictionary, i.e. captions['en']).
  caption_obj   = yt.captions.get_by_language_code('en')


/tmp/ipykernel_28718/986113750.py:27: DeprecationWarning: Call to deprecated function get_by_language_code (This object can be treated as a dictionary, i.e. captions['en']).
  caption_obj   = yt.captions.get_by_language_code('en')


...

Failed to process URL https://youtube.com/shorts/6qHXenlRDH4: 6qHXenlRDH4 has an unknown error, check logs for more info [Status: ERROR] [Reason: This video is no longer available due to a privacy claim by a third party]


/tmp/ipykernel_28718/986113750.py:27: DeprecationWarning: Call to deprecated function get_by_language_code (This object can be treated as a dictionary, i.e. captions['en']).
  caption_obj   = yt.captions.get_by_language_code('en')


/tmp/ipykernel_28718/986113750.py:27: DeprecationWarning: Call to deprecated function get_by_language_code (This object can be treated as a dictionary, i.e. captions['en']).
  caption_obj   = yt.captions.get_by_language_code('en')


/tmp/ipykernel_28718/986113750.py:27: DeprecationWarning: Call to deprecated function get_by_language_code (This object can be treated as a dictionary, i.e. captions['en']).
  caption_obj   = yt.captions.get_by_language_code('en')


Finished processing.████████████████████████████| 100.0%
Successfully processed: 281 videos.
Failed to process: 11 videos. Check 'video_errors.csv' for details.


In [ ]:
metadata_df.to_csv('video_metadata.csv', index=False, encoding='utf-8-sig')
failed_df.to_csv('video_errors.csv', index=False, encoding='utf-8-sig')



In [9]:
# Count unique in youtube_url
df['youtube_url'].nunique()


292